# LoRA Fine-tuning Experiment Grid

Runs multiple training configurations, evaluates each on the held-out test set, and saves a summary to `results/experiment_results.json`.

In [ ]:
import json
import sys
from pathlib import Path
from types import SimpleNamespace

import torch
import yaml

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from torch.utils.data import DataLoader

from src.dataset import make_splits
from src.model import load_model_with_lora
from src.train import evaluate, run_experiment
from src.utils import setup_logging

setup_logging()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# Base config — load from file, then override per experiment
with open("../configs/config.yaml") as f:
    BASE_CFG_DICT = yaml.safe_load(f)

def make_cfg(overrides: dict) -> SimpleNamespace:
    """Deep-merge overrides into a fresh copy of BASE_CFG_DICT."""
    import copy
    d = copy.deepcopy(BASE_CFG_DICT)
    for section, params in overrides.items():
        d[section].update(params)

    def to_ns(obj):
        if isinstance(obj, dict):
            ns = SimpleNamespace()
            for k, v in obj.items():
                setattr(ns, k, to_ns(v))
            return ns
        return obj

    return to_ns(d)

In [ ]:
# Experiment grid
# for local run use only one experiment
if DEVICE == "cpu":
    EXPERIMENTS = [
         {
        "name": "lr1e-4_r16-small",
        "overrides": {"training": {"learning_rate": 1e-4, "epochs": 5, "batch_size": 2},
                      "lora":     {"r": 8, "lora_alpha": 32}},
    },
    ]
else:
    EXPERIMENTS = [
        {
            "name": "lr1e-4_r16",
            "overrides": {"training": {"learning_rate": 1e-4, "epochs": 5, "batch_size": 8},
                        "lora":     {"r": 16, "lora_alpha": 32}},
        },
        {
            "name": "lr1e-5_r16",
            "overrides": {"training": {"learning_rate": 1e-5, "epochs": 5, "batch_size": 8},
                        "lora":     {"r": 16, "lora_alpha": 32}},
        },
        {
            "name": "lr1e-4_r8",
            "overrides": {"training": {"learning_rate": 1e-4, "epochs": 5, "batch_size": 8},
                        "lora":     {"r": 8,  "lora_alpha": 16}},
        },
        {
            "name": "lr1e-5_r8",
            "overrides": {"training": {"learning_rate": 1e-5, "epochs": 2, "batch_size": 2},
                        "lora":     {"r": 8,  "lora_alpha": 16}},
        },
]

In [ ]:
# Build datasets once — reused across all experiments
base_cfg = make_cfg({})
_, processor = load_model_with_lora(base_cfg)

rgb_dir   = PROJECT_ROOT / base_cfg.data.raw_dir / "rgb"
depth_dir = PROJECT_ROOT / base_cfg.data.raw_dir / "depth"
split_csv = PROJECT_ROOT / base_cfg.data.split_csv

train_ds, val_ds, test_ds = make_splits(
    rgb_dir, depth_dir, processor,
    split_csv=split_csv,
    train_ratio=base_cfg.data.train_ratio,
    val_ratio=base_cfg.data.val_ratio,
    test_ratio=base_cfg.data.test_ratio,
    seed=base_cfg.data.seed,
)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")

In [ ]:
# Baseline evaluation — off-the-shelf model, no fine-tuning
base_model, _ = load_model_with_lora(base_cfg)
base_model = base_model.to(DEVICE).eval()

test_loader = DataLoader(test_ds, batch_size=base_cfg.training.batch_size, shuffle=False, num_workers=2)
baseline_metrics = evaluate(base_model, test_loader, DEVICE) # run inference on test set and evaluate metrics

print("--- Baseline metrics (pre-finetuning) ---")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.4f}")

del base_model  # free memory before training

In [ ]:
# Run all experiments — baseline is the first entry
all_results = [{"name": "baseline", "config": {}, "history": [], "test_metrics": baseline_metrics}]

for exp in EXPERIMENTS:
    cfg = make_cfg(exp["overrides"])
    result = run_experiment(exp["name"], cfg, train_ds, val_ds, test_ds, PROJECT_ROOT, DEVICE)
    result["config"] = exp["overrides"]
    all_results.append(result)

out_path = PROJECT_ROOT / "results" / "experiment_results.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(all_results, indent=2))
print(f"\nSaved to {out_path}")

In [ ]:
# Summary table
print(f"{'Name':<22} {'LR':<10} {'LoRA r':<8} {'abs_rel':<12} {'rmse':<12} {'delta1'}")
print("-" * 72)
for r in all_results:
    cfg_train = r["config"].get("training", {})
    cfg_lora  = r["config"].get("lora", {})
    tm = r["test_metrics"]
    lr   = cfg_train.get("learning_rate", "-")
    rank = cfg_lora.get("r", "-")
    print(f"{r['name']:<22} {str(lr):<10} {str(rank):<8} {tm['abs_rel']:<12.4f} {tm['rmse']:<12.4f} {tm['delta1']:.4f}")